# Notebook 05 - Webster signal timing

This notebook applies Webster's signal-timing formulas to selected intersections using flows computed in the assignment notebooks. The calculation is schematic: it does not use observed signal phases, turning counts, saturation-flow measurements, or field calibration.

For a signal with cycle length $C$, total lost time $L$, and critical flow ratios $y_i=q_i/s_i$, Webster's cycle formula is

$$C_0=\frac{1.5L+5}{1-Y},\qquad Y=\sum_i y_i.$$

The effective green time assigned to phase $i$ is

$$g_i=(C_0-L)\frac{y_i}{Y},$$

when $Y>0$. These formulas require $Y<1$ for the unconstrained cycle expression to be finite.

The delay expression used below is Webster's approximate average delay formula. Its inputs are synthetic or inferred in this notebook, so the computed delays should be read as model outputs for the constructed instance.


In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

from src.graph_utils import load_graph

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Libraries loaded.')

## 1. Load inputs

The notebook loads assignment results from Notebook 04 and the enriched road graph.


In [ ]:
with open(PROCESSED_DIR / 'results_poa.pkl', 'rb') as fh:
    res = pickle.load(fh)

x_ue       = res['x_ue']
t0         = res['t0']
cap        = res['cap']
edges_list = res['edges_list']
poa        = res['poa']

G = load_graph(RAW_DIR / 'chapinero_drive_enriched.graphml')
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print(f'Loaded graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'PoA del sistema: {poa:.5f}')

## 2. Select candidate signalized intersections

A node is treated as signalizable when it has sufficient graph degree and at least one incoming edge with positive modeled UE flow. This is a graph-based rule, not a field inventory of traffic signals.


In [ ]:
# Flows UE by edge en un diccionario
flow_dict = {edges_list[i]: x_ue[i] for i in range(len(edges_list))}

# Nodes with total degree >= 3
candidate_nodes = [
    n for n in G.nodes()
    if G.in_degree(n) + G.out_degree(n) >= 3
]

# Filter: at least one incoming edge with flow > 10 veh/h
signal_nodes = []
for n in candidate_nodes:
    incoming_flows = [
        flow_dict.get((u, n, k), 0)
        for u in G.predecessors(n)
        for k in G[u][n]
    ]
    if max(incoming_flows, default=0) > 10:
        signal_nodes.append(n)

print(f'Nodes candidatos (grado ≥ 3)    : {len(candidate_nodes):,}')
print(f'Signalizable intersections   : {len(signal_nodes):,}')

## 3. Construct phase groups

Incoming approaches are grouped into simplified phases using their approach angles. This approximation ignores turning movements, lane groups, pedestrian phases, and existing signal plans.


In [ ]:
@dataclass
class Phase:
    """A signal phase with its incoming movements."""
    id: int
    edges: List[Tuple]           # edges with green in this phase
    flow: float = 0.0            # critical flow de la fase (veh/h)
    saturation: float = 1800.0   # saturation flow (veh/h)
    lost_time: float = 4.0       # lost time per phase (s)

    @property
    def y(self) -> float:
        """Critical-flow ratio: y = q / s."""
        return self.flow / self.saturation if self.saturation > 0 else 0


@dataclass
class Intersection:
    """Signalized intersection with its signal plan."""
    node_id: int
    phases: List[Phase]
    cycle: float = 0.0           # optimal cycle Webster (s)
    greens: List[float] = field(default_factory=list)  # times verdes (s)
    delay: float = 0.0           # average Webster delay (s/veh)
    Y: float = 0.0               # sum of critical ratios
    L: float = 0.0               # lost time total


def assign_phases(node, G, flow_dict, n_phases=2):
    """Group incoming edges into n_phases by approach angle."""
    predecessors = list(G.predecessors(node))
    if not predecessors:
        return []

    node_data = G.nodes[node]
    node_x, node_y = node_data.get('x', 0), node_data.get('y', 0)

    # Compute each predecessor angle
    angles = []
    for pred in predecessors:
        pred_data = G.nodes[pred]
        dx = node_x - pred_data.get('x', 0)
        dy = node_y - pred_data.get('y', 0)
        angle = np.degrees(np.arctan2(dy, dx)) % 360
        angles.append(angle)

    # Agrupar en n_phases sectores angulares
    sector_size = 360 / n_phases
    phases = []
    for ph_id in range(n_phases):
        lo = ph_id * sector_size
        hi = lo + sector_size
        phase_edges = []
        phase_flow  = 0
        for i, (pred, angle) in enumerate(zip(predecessors, angles)):
            if lo <= angle < hi:
                for k in G[pred][node]:
                    e = (pred, node, k)
                    f = flow_dict.get(e, 0)
                    phase_edges.append(e)
                    phase_flow = max(phase_flow, f)  # critical flow = maximum
        if phase_edges:  # only add phases with movements
            phases.append(Phase(id=ph_id, edges=phase_edges, flow=phase_flow))

    # If angular assignment is not useful, use two phases with all approaches
    if len(phases) < 2:
        mid = len(predecessors) // 2
        phases = []
        for ph_id, preds_group in enumerate([predecessors[:mid or 1], predecessors[mid or 1:]]):
            ph_edges, ph_flow = [], 0
            for pred in preds_group:
                for k in G[pred][node]:
                    e = (pred, node, k)
                    f = flow_dict.get(e, 0)
                    ph_edges.append(e)
                    ph_flow = max(ph_flow, f)
            phases.append(Phase(id=ph_id, edges=ph_edges, flow=ph_flow))

    return phases


print('Modelo de fases definido.')

## 4. Define Webster formulas

The functions below compute cycle length, effective greens, and approximate average delay for the simplified phase representation.


In [ ]:
def webster_cycle(phases: List[Phase],
                  C_min: float = 30,
                  C_max: float = 180) -> float:
    """
    Webster optimal cycle: C0 = (1.5*L + 5) / (1 - Y)
    Clipeado en [C_min, C_max].
    """
    L = sum(ph.lost_time for ph in phases)
    Y = sum(ph.y for ph in phases)
    if Y >= 1.0:
        return C_max   # oversaturated intersection
    if Y <= 0:
        return C_min
    C0 = (1.5 * L + 5) / (1 - Y)
    return float(np.clip(C0, C_min, C_max))


def webster_greens(phases: List[Phase], cycle: float) -> List[float]:
    """
    Effective green time by phase:
    g_i = (C - L) * y_i / Y
    """
    L = sum(ph.lost_time for ph in phases)
    Y = sum(ph.y for ph in phases)
    effective_green = cycle - L
    if Y <= 0 or effective_green <= 0:
        return [effective_green / len(phases)] * len(phases)
    return [effective_green * ph.y / Y for ph in phases]


def webster_delay(q: float, s: float, g: float, C: float,
                  l: float = 4.0) -> float:
    """
    Average delay per vehicle (Webster 1958, s/veh).
    q = flow de demand (veh/s)
    s = saturation flow (veh/s)
    g = green time (s)
    C = cycle (s)
    """
    q_s   = q / 3600       # veh/s
    s_s   = s / 3600       # veh/s
    lam   = g / C          # green fraction
    x     = q_s / (s_s * lam) if (s_s * lam) > 0 else 0.99  # degree of saturation
    x     = min(x, 0.99)   # avoid division by zero near saturation

    # Term 1: uniform delay
    d1 = C * (1 - lam) ** 2 / (2 * (1 - lam * x))
    # Term 2: random delay
    d2 = x ** 2 / (2 * q_s * (1 - x)) if (1 - x) > 0 else 0
    # Term 3: Webster empirical correction
    d3 = 0.65 * (C / (q_s ** 2 + 1e-9)) ** (1/3) * x ** (2 + 5 * lam)

    return max(d1 + d2 - d3, 0)


print('Funciones de Webster definidas.')

## 5. Apply Webster formulas

The calculation is applied independently at each selected node. Interactions between adjacent signals are not modeled.


In [ ]:
intersections: List[Intersection] = []

for node in signal_nodes:
    phases = assign_phases(node, G, flow_dict, n_phases=2)
    if not phases:
        continue

    C0  = webster_cycle(phases)
    gs  = webster_greens(phases, C0)
    L   = sum(ph.lost_time for ph in phases)
    Y   = sum(ph.y for ph in phases)

    # Flow-weighted average delay
    total_flow  = sum(ph.flow for ph in phases)
    total_delay = 0.0
    for ph, g in zip(phases, gs):
        d = webster_delay(ph.flow, ph.saturation, g, C0, ph.lost_time)
        total_delay += d * ph.flow
    avg_delay = total_delay / total_flow if total_flow > 0 else 0

    inter = Intersection(
        node_id=node,
        phases=phases,
        cycle=C0,
        greens=gs,
        delay=avg_delay,
        Y=Y,
        L=L,
    )
    intersections.append(inter)

print(f'Intersecciones procesadas: {len(intersections)}')

## 6. Summarize signal plans

The table summarizes cycle lengths, critical flow ratios, modeled delays, and modeled flows for the selected nodes.


In [ ]:
df_signals = pd.DataFrame([{
    'node':    inter.node_id,
    'n_fases': len(inter.phases),
    'Y':       inter.Y,
    'L':       inter.L,
    'cycle':   inter.cycle,
    'delay':  inter.delay,
    'flow_total': sum(ph.flow for ph in inter.phases),
    'lat':     G.nodes[inter.node_id].get('y', np.nan),
    'lon':     G.nodes[inter.node_id].get('x', np.nan),
} for inter in intersections])

print('Signal-plan statistics:')
display(df_signals[['cycle','delay','Y','flow_total']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Optimal cycle
ax = axes[0, 0]
ax.hist(df_signals['cycle'], bins=25, color='#3498db', edgecolor='white', linewidth=0.4)
ax.axvline(df_signals['cycle'].meann(), color='red', linestyle='--',
           label=f'Median={df_signals["cycle"].meann():.0f}s')
ax.set_xlabel('Optimal cycle Webster (s)', fontsize=11)
ax.set_ylabel('Intersecciones', fontsize=11)
ax.set_title('Distribution of optimal cycles')
ax.legend()

# Average delay
ax = axes[0, 1]
ax.hist(df_signals['delay'].clip(upper=120), bins=25, color='#e74c3c', edgecolor='white', linewidth=0.4)
ax.axvline(df_signals['delay'].meann(), color='navy', linestyle='--',
           label=f'Median={df_signals["delay"].meann():.1f}s')
ax.set_xlabel('Average Webster delay (s/veh)', fontsize=11)
ax.set_ylabel('Intersecciones', fontsize=11)
ax.set_title('Distribution of intersection delays')
ax.legend()

# Y (degree of saturation agregado)
ax = axes[1, 0]
ax.hist(df_signals['Y'].clip(upper=1), bins=25, color='#2ecc71', edgecolor='white', linewidth=0.4)
ax.axvline(0.85, color='red', linestyle='--', linewidth=1.2, label='Critical threshold Y=0.85')
ax.set_xlabel('Y (suma de razones de critical flow)', fontsize=11)
ax.set_ylabel('Intersecciones', fontsize=11)
ax.set_title('Distribution of Y (aggregate saturation)')
ax.legend()

# Ciclo vs delay
ax = axes[1, 1]
sc = ax.scatter(df_signals['cycle'], df_signals['delay'].clip(upper=120),
                c=df_signals['Y'], cmap='RdYlGn_r', alpha=0.6, s=20)
plt.colorbar(sc, ax=ax, label='Y (saturation)')
ax.set_xlabel('Optimal cycle (s)', fontsize=11)
ax.set_ylabel('Demora (s/veh)', fontsize=11)
ax.set_title('Cycle vs. delay (color = saturation Y)')

plt.suptitle('Webster signal plans — Chapinero, Bogota', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_stats.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Identify high-delay and high-saturation nodes

The rankings are computed from the simplified Webster model. They are diagnostic outputs, not validated operational recommendations.


In [ ]:
top_delay = df_signals.nlargest(10, 'delay')[['node','cycle','Y','delay','flow_total']]
top_Y     = df_signals.nlargest(10, 'Y')[['node','cycle','Y','delay','flow_total']]

print('Top 10 intersections por DEMORA (s/veh):')
display(top_delay.round(2))

print('\nTop 10 intersections by SATURATION (Y):')
display(top_Y.round(4))

## 8. Plot phase diagrams

The diagrams show the simplified green and lost-time allocation for selected nodes.


In [ ]:
def plot_signal_diagram(inter: Intersection, ax, title=''):
    """Draw the phase diagram (green/red/yellow) for an intersection."""
    C   = inter.cycle
    gs  = inter.greens
    yl  = 3   # yellow (s)
    colors_ph = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

    current = 0
    for ph_id, (ph, g) in enumerate(zip(inter.phases, gs)):
        color = colors_ph[ph_id % len(colors_ph)]
        # Green
        ax.barh(ph_id, g, left=current, color='#2ecc71', edgecolor='white', linewidth=0.5)
        ax.text(current + g / 2, ph_id, f'{g:.0f}s', ha='center', va='center',
                fontsize=8, color='black', fontweight='bold')
        current += g
        # Yellow
        ax.barh(ph_id, yl, left=current, color='#f1c40f', edgecolor='white', linewidth=0.5)
        current += yl
        # Lost time (all red)
        lost = ph.lost_time - yl
        if lost > 0:
            ax.barh(ph_id, lost, left=current, color='#c0392b', edgecolor='white', linewidth=0.5)
            current += lost

    # Red for phases without green
    ax.set_yticks(range(len(inter.phases)))
    ax.set_yticklabels([f'Phase {ph.id}\ny={ph.y:.3f}' for ph in inter.phases], fontsize=9)
    ax.set_xlabel('Time in cycle (s)', fontsize=9)
    ax.set_xlim(0, C)
    ax.set_title(f'{title}\nC={C:.0f}s  Y={inter.Y:.3f}  dem={inter.delay:.1f}s/veh',
                 fontsize=9)

    patches = [
        mpatches.Patch(color='#2ecc71', label='Green'),
        mpatches.Patch(color='#f1c40f', label='Yellow'),
        mpatches.Patch(color='#c0392b', label='All red'),
    ]
    ax.legend(handles=patches, fontsize=7, loc='lower right')


# Plot the 6 most critical intersections by delay
critical = sorted(intersections, key=lambda x: x.delay, reverse=True)[:6]

fig, axes = plt.subplots(3, 2, figsize=(13, 10))
for ax, inter in zip(axes.flatten(), critical):
    plot_signal_diagram(inter, ax, title=f'Nodo {inter.node_id}')

plt.suptitle('Webster signal plans - top 6 critical intersections\nChapinero, Bogota',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Map modeled cycles and delays

The map displays outputs of the simplified signal model on the road graph.


In [ ]:
# Assign graph attributes for visualization
for inter in intersections:
    G.nodes[inter.node_id]['webster_cycle'] = inter.cycle
    G.nodes[inter.node_id]['webster_delay'] = inter.delay
    G.nodes[inter.node_id]['webster_Y']     = inter.Y

# Node colors and sizes
signal_set = {inter.node_id for inter in intersections}
node_sizes  = []
node_colors = []
cmap_delay  = plt.get_cmap('RdYlGn_r')
max_delay   = df_signals['delay'].quantile(0.95)

for n in G.nodes():
    if n in signal_set:
        delay = G.nodes[n].get('webster_delay', 0)
        norm_d = min(delay / max_delay, 1.0)
        node_colors.append(cmap_delay(norm_d))
        node_sizes.append(30 + 60 * norm_d)
    else:
        node_colors.append('#2a2a4a')
        node_sizes.append(3)

fig, ax = ox.plot_graph(
    G,
    node_color=node_colors,
    node_size=node_sizes,
    edge_color='#3a3a5a',
    edge_linewidth=0.6,
    bgcolor='#1a1a2e',
    figsize=(12, 10),
    show=False,
    close=False,
)

sm = plt.cm.ScalarMappable(cmap=cmap_delay,
                            norm=mcolors.Normalize(vmin=0, vmax=max_delay))
sm.set_array([])
plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.55,
             label='Demora Webster (s/veh)')

ax.set_title('Delay at signalized intersections (Webster formula)\nChapinero, Bogota',
             color='white', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_delay_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Compare with a fixed-cycle reference

The comparison uses a simple fixed-cycle reference. It is not a before-and-after evaluation of real signal timing.


In [ ]:
# Compare Webster delay against delay with a fixed reference cycle (C=90s, green=50%)
C_ref = 90.0
g_ref_frac = 0.50

rows_impact = []
for inter in intersections:
    for ph, g_opt in zip(inter.phases, inter.greens):
        if ph.flow < 1:
            continue
        g_ref = C_ref * g_ref_frac
        d_opt = webster_delay(ph.flow, ph.saturation, g_opt, inter.cycle, ph.lost_time)
        d_ref = webster_delay(ph.flow, ph.saturation, g_ref, C_ref, ph.lost_time)
        rows_impact.append({
            'node':    inter.node_id,
            'flow':    ph.flow,
            'd_opt':   d_opt,
            'd_ref':   d_ref,
            'saving':  d_ref - d_opt,        # s/veh
            'saving_total': (d_ref - d_opt) * ph.flow,  # veh·s/h
        })

df_impact = pd.DataFrame(rows_impact)
total_saving = df_impact['saving_total'].sum()
avg_saving   = df_impact['saving'].mean()

print('=' * 55)
print('   WEBSTER OPTIMIZATION IMPACT')
print('=' * 55)
print(f'  Intersecciones optimizadas  : {len(intersections):,}')
print(f'  Mean optimal cycle          : {df_signals["cycle"].mean():.1f} s')
print(f'  Mean optimal delay         : {df_signals["delay"].mean():.1f} s/veh')
print(f'  Demora mean fixed cycle 90s : {df_impact["d_ref"].mean():.1f} s/veh')
print(f'  Mean saving             : {avg_saving:.1f} s/veh')
print(f'  Ahorro total del sistema    : {total_saving:,.0f} veh·s/h')
print(f'                              = {total_saving/3600:,.1f} veh·h/h')
print('=' * 55)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_impact['d_ref'], df_impact['d_opt'],
           c=df_impact['flow'], cmap='plasma', alpha=0.5, s=15)
lim = max(df_impact[['d_ref','d_opt']].max())
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='Sin cambio')
ax.set_xlabel('Fixed-cycle 90s delay (s/veh)', fontsize=11)
ax.set_ylabel('Optimal Webster delay (s/veh)', fontsize=11)
ax.set_title('Delay: fixed cycle vs. optimized Webster\n(points below diagonal = improvement)')
ax.legend()
sm = plt.cm.ScalarMappable(cmap='plasma')
sm.set_array(df_impact['flow'])
plt.colorbar(sm, ax=ax, label='Flujo (veh/h)')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary quantities

The printed summary records outputs of the constructed numerical instance. It should not be interpreted as a validated estimate of network performance.


In [ ]:
cost_ue  = res['cost_ue']
cost_so  = res['cost_so']

print('=' * 65)
print('   SUMMARY - CHAPINERO TRAFFIC OPTIMIZATION')
print('=' * 65)
print()
print('1. RED VIAL')
print(f'   Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}')
print(f'   O-D pairs modelados: {len(res["od_pairs"])}')
print()
print('2. EQUILIBRIO DE WARDROP (UE)')
print(f'   Total cost: {cost_ue:,.0f} veh·s')
print(f'   Tiempo prom: {cost_ue/sum(d for _,_,d in res["od_pairs"]):.1f} s/veh')
print()
print('3. SOCIAL OPTIMUM (SO)')
print(f'   Total cost: {cost_so:,.0f} veh·s')
print(f'   Ahorro vs UE: {(cost_ue-cost_so)/3600:,.1f} veh·h/h  ({(poa-1)*100:.2f}%)')
print()
print('4. PRICE OF ANARCHY')
print(f'   PoA = {poa:.5f}  (Roughgarden bound: {res["theoretical_bound"]:.4f})')
print(f'   The system is {(poa-1)*100:.2f}% less efficient than the centralized optimum')
print()
print('5. SIGNALS - WEBSTER FORMULA')
print(f'   Intersecciones optimizadas: {len(intersections):,}')
print(f'   Mean optimal cycle        : {df_signals["cycle"].mean():.1f} s')
print(f'   Demora mean optimizada   : {df_signals["delay"].mean():.1f} s/veh')
print(f'   Saving vs fixed 90s cycle  : {avg_saving:.1f} s/veh  ({total_saving/3600:,.1f} veh·h/h)')
print('=' * 65)

## 12. Save results

The saved files contain modeled signal quantities for this notebook's assumptions.

Limitations of this notebook:
- Signal phases are inferred from graph geometry, not observed controller data.
- Turning movements and lane groups are not modeled.
- Saturation-flow assumptions are not calibrated.
- Coordination between adjacent signals is not represented.


In [ ]:
results_final = {
    **res,
    'intersections': intersections,
    'df_signals':    df_signals,
    'df_impact':     df_impact,
    'total_saving':  total_saving,
    'avg_saving':    avg_saving,
}

out_path = PROCESSED_DIR / 'results_final.pkl'
with open(out_path, 'wb') as fh:
    pickle.dump(results_final, fh)

# Export signal table to CSV
df_signals.to_csv(PROCESSED_DIR / 'signal_plans.csv', index=False)

print(f'Resultados finales guardados en: {out_path}')
print(f'Signal-plan table exported to: {PROCESSED_DIR}/signal_plans.csv')
print('\nProyecto completado.')